# Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt

In [3]:
base_dir = '/content/drive/MyDrive/Grocery-Orders-Analysis'
data_dir = Path(base_dir) / 'data'

d = {
    file.stem: pd.read_csv(file)
    for file in data_dir.glob('*.csv')
}



In [4]:
d.keys()

dict_keys(['aisles', 'departments', 'order_products__prior', 'order_products__train', 'products', 'orders'])

In [5]:
print(d['order_products__prior'].isna().sum(),
d['order_products__train'].isna().sum(),sep="\n")

order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64
order_id             0
product_id           0
add_to_cart_order    0
reordered            0
dtype: int64


In [6]:
df = pd.concat([d['order_products__prior'],d['order_products__train']],ignore_index=True)
df.isna().sum()

,0
order_id,0
product_id,0
add_to_cart_order,0
reordered,0


In [7]:
orders = d['orders'].copy()
orders.drop(columns=['eval_set'],inplace=True)

In [8]:
orders['days_since_prior_order'].fillna(-1,inplace=True)
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 6 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   order_number            int64  
 3   order_dow               int64  
 4   order_hour_of_day       int64  
 5   days_since_prior_order  float64
dtypes: float64(1), int64(5)
memory usage: 156.6 MB


/tmp/ipykernel_26218/694218418.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  orders['days_since_prior_order'].fillna(-1,inplace=True)


In [9]:
df = pd.merge(df,orders,how='left',on='order_id')
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 9 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   order_number            int64  
 6   order_dow               int64  
 7   order_hour_of_day       int64  
 8   days_since_prior_order  float64
dtypes: float64(1), int64(8)
memory usage: 2.3 GB


In [10]:
df.isna().sum()
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2,33120,1,1,202279,3,5,9,8.0
1,2,28985,2,1,202279,3,5,9,8.0
2,2,9327,3,0,202279,3,5,9,8.0
3,2,45918,4,1,202279,3,5,9,8.0
4,2,30035,5,0,202279,3,5,9,8.0


In [11]:
df = pd.merge(df,d['products'],how='left',on='product_id')
print(df.head(),df.info(),df.isna().sum(),sep="\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33819106 entries, 0 to 33819105
Data columns (total 12 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   product_id              int64  
 2   add_to_cart_order       int64  
 3   reordered               int64  
 4   user_id                 int64  
 5   order_number            int64  
 6   order_dow               int64  
 7   order_hour_of_day       int64  
 8   days_since_prior_order  float64
 9   product_name            object 
 10  aisle_id                int64  
 11  department_id           int64  
dtypes: float64(1), int64(10), object(1)
memory usage: 3.0+ GB
   order_id  product_id  add_to_cart_order  reordered  user_id  order_number  \
0         2       33120                  1          1   202279             3   
1         2       28985                  2          1   202279             3   
2         2        9327                  3          0   202279  

In [ ]:
df = pd.merge(df,d['aisles'],how='left',on='aisle_id')
df = pd.merge(df,d['departments'],how='left',on='department_id')
print(df.head(),df.info(),df.isna().sum(),sep="\n")